In [ ]:
"""
# Notebook Documentation: Simulation Plan for Verifying Value-Weighted Segmentedness Estimation

## Objective
This notebook verifies the paper’s theory-driven estimation workflow for **value-weighted segmentedness** using simulation, with emphasis on whether **97 Phase-2 connectivity tests** are sufficient under the stated worst-case design target.

## Network and Asset-Tier Setup
- Three asset tiers (blocks): **critical, sensitive, commodity**
- Total nodes: `n = 100000`
- Pyramid tier fractions:
  - `f1 = 0.05` (critical)
  - `f2 = 0.25` (sensitive)
  - `f3 = 0.70` (commodity)
- Value model: geometric with `rho = 0.1`
  - `w1 = 1`
  - `w2 = rho`
  - `w3 = rho^2`

## Connectivity Model (Policy-Permitted Pair Probability)
- Connectivity means **policy-permitted communication probability** (paper-consistent).
- One shared within-tier probability: `p_in`
- One shared cross-tier probability per scenario: `p_out`
- Sweep settings:
  - `p_in in {0.10, 0.15, ..., 0.50}` (inclusive)
  - `p_out in {0.10, 0.15, 0.20}` (one chart per `p_out`)

## Ground-Truth Metric
- True value-weighted segmentedness `S_w_true` is computed **analytically** from:
  1. Large-n class proportions (paper approximation),
  2. Tier weights (`w_k`),
  3. Class segmentedness implied by (`p_in`, `p_out`) under the tier-pair structure.

## Estimation Procedure (Paper-Style Two Phase)
1. **Phase 1 (cheap tier inspection)**  
   - Sample nodes with replacement.
   - Default sample size: `N1 = 625` (tunable parameter).
   - Estimate tier fractions, then class proportions, then plug-in class weights `a_hat`.

2. **Phase 2 (expensive connectivity tests)**  
   - Exactly `M = 97` connectivity tests total.
   - Allocate by paper method using `a_hat`:
     - integer counts,
     - minimum 1 test per positive-weight class,
     - residual adjustment to ensure counts sum to 97.
   - Sampling is with replacement.
   - Class draws are done by **direct class-conditioned sampling** (not rejection sampling).
   - Compute paper plug-in estimator `S_w_hat`.

## Monte Carlo Design
- Trials per setting: `T = 1000`
- Fixed global random seed set once in notebook (tunable for reproducibility).

## Reported Outputs
1. **Summary table** (first output block), per (`p_out`, `p_in`) setting:
   - `true_Sw`
   - `mean_hat_Sw`
   - `bias = mean_hat_Sw - true_Sw`
   - empirical CI low (2.5th percentile of trial estimates)
   - empirical CI high (97.5th percentile of trial estimates)

2. **Charts** (separate code block):
   - Three charts (one per `p_out`)
   - x-axis: `p_in`
   - plotted series:
     - analytic true `S_w`
     - empirical mean estimate
     - empirical 95% interval (2.5th–97.5th percentiles across 1000 trials)

## Conventions
- All sampling is with replacement (paper-consistent).
- Large-n approximation is used for class proportions (paper-consistent).
- No allocation-strategy comparison is included in this version (may be added later).
"""

In [ ]:
# ============================================================
# Value-Weighted Segmentedness Simulation (Paper-Style, 2-Phase)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# Global tunable parameters
# -----------------------------
SEED = 97  # fixed global seed (tunable)
rng = np.random.default_rng(SEED)

n = 100_000
f_true = np.array([0.05, 0.25, 0.70])   # (critical, sensitive, commodity)
rho = 0.1
w = np.array([1.0, rho, rho**2])        # geometric weights

N1 = 625          # Phase 1 sample size (tunable)
M = 97            # Phase 2 connectivity tests (fixed by spec)
TRIALS = 1000

p_in_values = np.round(np.arange(0.10, 0.50 + 1e-12, 0.05), 2)   # inclusive
p_out_values = [0.10, 0.15, 0.20]

In [ ]:
# -----------------------------
# Utility functions
# -----------------------------
def class_probs_from_f_large_n(f):
    """
    Large-n approximation from paper for K=3:
      pi1 = 1 - (1-f1)^2
      pi3 = f3^2
      pi2 = 1 - pi1 - pi3
    """
    f1, f2, f3 = f
    pi1 = 1.0 - (1.0 - f1)**2
    pi3 = f3**2
    pi2 = 1.0 - pi1 - pi3
    pi = np.array([pi1, pi2, pi3], dtype=float)
    return pi

def class_segmentedness_from_pin_pout(p_in, p_out, f):
    """
    For class k = min(tier_u, tier_v), with tiers 1..3:
      Class 1: pairs (1,1), (1,2), (1,3)
      Class 2: pairs (2,2), (2,3)
      Class 3: pairs (3,3)

    Under independent endpoint draws (large-n approximation):
      P(tier=i, tier=j) for unordered pair is:
        i==j: f_i^2
        i<j : 2 f_i f_j

    Connectivity probability:
      within-tier pairs: p_in
      between-tier pairs: p_out

    Class segmentedness S_k = 1 - E[connectivity | class k].
    """
    f1, f2, f3 = f
    pi = class_probs_from_f_large_n(f)
    pi1, pi2, pi3 = pi

    # class-1 mass contributions
    m11 = f1**2
    m12 = 2*f1*f2
    m13 = 2*f1*f3
    conn1 = (m11*p_in + (m12+m13)*p_out) / pi1
    S1 = 1.0 - conn1

    # class-2 mass contributions
    m22 = f2**2
    m23 = 2*f2*f3
    conn2 = (m22*p_in + m23*p_out) / pi2
    S2 = 1.0 - conn2

    # class-3
    conn3 = p_in
    S3 = 1.0 - conn3

    return np.array([S1, S2, S3], dtype=float)

def true_weighted_segmentedness(p_in, p_out, f, w):
    pi = class_probs_from_f_large_n(f)
    S = class_segmentedness_from_pin_pout(p_in, p_out, f)
    num = np.sum(w * pi * S)
    den = np.sum(w * pi)
    return num / den

def sample_phase1_fhat(N1, f_true, rng):
    """
    Sample node tiers with replacement; multinomial counts.
    """
    counts = rng.multinomial(N1, f_true)
    return counts / N1

def pihat_from_fhat_large_n(fhat):
    return class_probs_from_f_large_n(fhat)

def ahat_from_pihat(pihat, w):
    raw = w * pihat
    den = np.sum(raw)
    if den <= 0:
        # Shouldn't happen with positive weights & valid probs, but safeguard:
        return np.ones_like(raw) / len(raw)
    return raw / den

def integer_allocation_with_floor(q, M, w):
    """
    Paper-style practical integer allocation:
      - round M*q
      - at least 1 test for each positive-weight class
      - adjust residual to sum exactly M

    Assumes M >= number of positive-weight classes.
    """
    q = np.array(q, dtype=float)
    K = len(q)
    pos = w > 0
    Kpos = np.sum(pos)
    if M < Kpos:
        raise ValueError("M is smaller than number of positive-weight classes.")

    # initial rounded allocation
    mk = np.rint(M * q).astype(int)

    # enforce floor of 1 for positive-weight classes
    mk[(pos) & (mk < 1)] = 1
    mk[~pos] = 0

    # adjust to exact total M
    diff = M - mk.sum()

    if diff > 0:
        # add to classes with largest fractional deficits (M*q - mk), positive weights only
        frac = M*q - mk
        candidates = np.where(pos)[0]
        order = candidates[np.argsort(frac[candidates])[::-1]]
        i = 0
        while diff > 0:
            mk[order[i % len(order)]] += 1
            diff -= 1
            i += 1
    elif diff < 0:
        # remove from classes with largest surpluses (mk - M*q), but keep floor>=1 for pos
        surplus = mk - M*q
        candidates = np.where(pos)[0]
        order = candidates[np.argsort(surplus[candidates])[::-1]]
        i = 0
        while diff < 0:
            idx = order[i % len(order)]
            if mk[idx] > 1:
                mk[idx] -= 1
                diff += 1
            i += 1
            # safety break (should not happen)
            if i > 10_000:
                raise RuntimeError("Allocation adjustment failed.")

    return mk

def simulate_class_y_disallowed(class_k, size, p_in, p_out, f, rng):
    """
    Direct class-conditioned sampling:
      Draw pair type within class according to large-n pair-mass decomposition,
      then sample disallowed Bernoulli = 1 - connectivity prob.
    class_k in {1,2,3}
    """
    f1, f2, f3 = f
    if class_k == 1:
        masses = np.array([f1**2, 2*f1*f2, 2*f1*f3], dtype=float)   # (1,1),(1,2),(1,3)
        probs = masses / masses.sum()
        # connectivity probs aligned with masses:
        conn = np.array([p_in, p_out, p_out], dtype=float)
    elif class_k == 2:
        masses = np.array([f2**2, 2*f2*f3], dtype=float)            # (2,2),(2,3)
        probs = masses / masses.sum()
        conn = np.array([p_in, p_out], dtype=float)
    elif class_k == 3:
        # only (3,3)
        probs = np.array([1.0], dtype=float)
        conn = np.array([p_in], dtype=float)
    else:
        raise ValueError("class_k must be 1,2,3")

    types = rng.choice(len(probs), size=size, p=probs)
    p_conn = conn[types]
    y_disallowed = rng.binomial(1, 1.0 - p_conn, size=size)  # 1 = disallowed
    return y_disallowed

def one_trial_estimate(p_in, p_out, f_true, w, N1, M, rng):
    # Phase 1
    fhat = sample_phase1_fhat(N1, f_true, rng)
    pihat = pihat_from_fhat_large_n(fhat)
    # numerical guard: clip and renormalize
    pihat = np.clip(pihat, 1e-12, None)
    pihat = pihat / pihat.sum()

    ahat = ahat_from_pihat(pihat, w)
    ahat = np.clip(ahat, 1e-12, None)
    ahat = ahat / ahat.sum()

    # integer allocation
    Mk = integer_allocation_with_floor(ahat, M, w)

    # Allocation assertions
    assert Mk.sum() == M, "Allocated tests must sum to M."
    assert np.all(Mk >= 0), "Allocated tests must be nonnegative."
    assert np.all(Mk[w > 0] >= 1), "Positive-weight classes must receive at least 1 test."

    # Phase 2: class-wise segmentedness estimates
    Shat = np.zeros(3, dtype=float)
    for k in range(1, 4):
        m_k = Mk[k-1]
        y = simulate_class_y_disallowed(k, m_k, p_in, p_out, f_true, rng)
        Shat[k-1] = y.mean()

    # Combined estimator (paper plug-in)
    num = np.sum(w * pihat * Shat)
    den = np.sum(w * pihat)
    Sw_hat = num / den

    # Estimator bounds
    assert np.all((Shat >= 0.0) & (Shat <= 1.0)), "Per-class segmentedness estimates must lie in [0,1]."
    assert 0.0 <= Sw_hat <= 1.0, "Estimated S_w must lie in [0,1]."
    
    return Sw_hat

In [ ]:
# -----------------------------
# Assertions: model and math sanity checks
# Insert after utility functions, before simulation loop
# -----------------------------
# Tier fractions
assert np.isclose(f_true.sum(), 1.0), "Tier fractions must sum to 1."
assert np.all(f_true > 0), "All tier fractions must be positive."

# Weights
assert np.all(w >= 0), "Weights must be nonnegative."
assert np.isclose(w[0], 1.0), "w1 must be normalized to 1."
assert np.isclose(w[1], rho) and np.isclose(w[2], rho**2), "Geometric weights mismatch."

# Sweep definitions
assert np.isclose(p_in_values[0], 0.10) and np.isclose(p_in_values[-1], 0.50), "p_in endpoints incorrect."
assert len(p_in_values) == 9, "p_in sweep should have 9 points (0.10 to 0.50 by 0.05)."
assert set(p_out_values) == {0.10, 0.15, 0.20}, "p_out values mismatch."

# Budget / phase settings
assert M == 97, "Phase 2 connectivity tests must be exactly 97."
assert N1 > 0, "Phase 1 sample size must be positive."
assert TRIALS == 1000, "Trials should be 1000 per setting per your spec."

# Large-n class proportion sanity
pi_true = class_probs_from_f_large_n(f_true)
assert np.all(pi_true > 0), "All class proportions should be positive under current pyramid."
assert np.isclose(pi_true.sum(), 1.0), "Class proportions must sum to 1."

# True S_w bounds sanity over whole grid
for p_out_chk in p_out_values:
    for p_in_chk in p_in_values:
        sw_chk = true_weighted_segmentedness(p_in_chk, p_out_chk, f_true, w)
        assert 0.0 <= sw_chk <= 1.0, "True S_w must lie in [0,1]."

In [ ]:
# -----------------------------
# Run simulation grid
# -----------------------------
rows = []
all_estimates = {}  # store for plotting: key=(p_out, p_in) -> np.array(TRIALS)

for p_out in p_out_values:
    for p_in in p_in_values:
        true_sw = true_weighted_segmentedness(p_in, p_out, f_true, w)

        est = np.empty(TRIALS, dtype=float)
        for t in range(TRIALS):
            est[t] = one_trial_estimate(p_in, p_out, f_true, w, N1, M, rng)

        # Trial-level assertions for this setting
        assert np.all(np.isfinite(est)), "Non-finite estimate encountered."
        assert np.all((est >= 0.0) & (est <= 1.0)), "Trial estimates must lie in [0,1]."

        mean_hat = est.mean()
        ci_low, ci_high = np.percentile(est, [2.5, 97.5])
        bias = mean_hat - true_sw

        rows.append({
            "p_out": p_out,
            "p_in": p_in,
            "true_Sw": true_sw,
            "mean_hat_Sw": mean_hat,
            "bias": bias,
            "emp_ci_low": ci_low,
            "emp_ci_high": ci_high
        })

        all_estimates[(p_out, p_in)] = est

results_df = pd.DataFrame(rows).sort_values(["p_out", "p_in"]).reset_index(drop=True)

In [ ]:
# Result table assertions
required_cols = {"p_out", "p_in", "true_Sw", "mean_hat_Sw", "bias", "emp_ci_low", "emp_ci_high"}
assert required_cols.issubset(results_df.columns), "Missing required result columns."
assert len(results_df) == len(p_out_values) * len(p_in_values), "Unexpected number of rows in results table."

# CI monotonicity and bounds
assert np.all(results_df["emp_ci_low"] <= results_df["emp_ci_high"]), "CI low must be <= CI high."
assert np.all((results_df["emp_ci_low"] >= 0) & (results_df["emp_ci_high"] <= 1)), "Empirical CI must lie in [0,1]."
assert np.all((results_df["true_Sw"] >= 0) & (results_df["true_Sw"] <= 1)), "true_Sw must lie in [0,1]."
assert np.all((results_df["mean_hat_Sw"] >= 0) & (results_df["mean_hat_Sw"] <= 1)), "mean_hat_Sw must lie in [0,1]."

In [ ]:
# -----------------------------
# Table output (as requested first)
# -----------------------------
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
# display(results_df)

# Optional: compact pivot table for quick scan
pivot_bias = results_df.pivot(index="p_in", columns="p_out", values="bias")
# display(pivot_bias)

In [ ]:
# Add empirical coverage indicator per setting
results_df["covered"] = (
    (results_df["true_Sw"] >= results_df["emp_ci_low"]) &
    (results_df["true_Sw"] <= results_df["emp_ci_high"])
)

# Optional numeric version (1/0)
results_df["coverage"] = results_df["covered"].astype(int)

# Reorder columns for display
results_df = results_df[
    ["p_out", "p_in", "true_Sw", "mean_hat_Sw", "bias", "emp_ci_low", "emp_ci_high", "covered", "coverage"]
].sort_values(["p_out", "p_in"]).reset_index(drop=True)

# display(results_df)

coverage_summary = results_df.groupby("p_out", as_index=False)["coverage"].mean()
coverage_summary = coverage_summary.rename(columns={"coverage": "coverage_rate_across_p_in_points"})
# display(coverage_summary)

In [ ]:
# ============================================================
# ONE-BLOCK TABLE CODE (replace prior table-generation code)
# Builds a LaTeX table with columns:
# p_out, p_in, analytical S_w, 2.5th percentile, 97.5th percentile, bias
#
# REQUIREMENTS:
#   - results_df must already exist with columns:
#       p_out, p_in, true_Sw, bias
#   - all_estimates must already exist as dict:
#       key: (p_out, p_in), value: np.array of trial estimates
# ============================================================

import numpy as np
import pandas as pd

# ---------- 1) Validate inputs ----------
required_cols = {"p_out", "p_in", "true_Sw", "bias"}
missing = required_cols - set(results_df.columns)
if missing:
    raise ValueError(f"results_df is missing required columns: {missing}")

if not isinstance(all_estimates, dict):
    raise ValueError("all_estimates must be a dict keyed by (p_out, p_in).")

# ---------- 2) Build summary rows ----------
rows = []
for _, r in results_df.iterrows():
    p_out = float(r["p_out"])
    p_in = float(r["p_in"])
    key = (p_out, p_in)

    if key not in all_estimates:
        raise KeyError(f"Missing key in all_estimates: {key}")

    est = np.asarray(all_estimates[key], dtype=float)
    if est.size == 0:
        raise ValueError(f"Empty trial-estimate array for key={key}")
    if not np.all(np.isfinite(est)):
        raise ValueError(f"Non-finite values found in trial estimates for key={key}")

    q025, q975 = np.percentile(est, [2.5, 97.5])

    rows.append({
        "p_out": p_out,
        "p_in": p_in,
        "analytical_Sw": float(r["true_Sw"]),
        "p2_5": float(q025),
        "p97_5": float(q975),
        "bias": float(r["bias"]),
    })

table_df = (
    pd.DataFrame(rows)
      .sort_values(["p_out", "p_in"])
      .reset_index(drop=True)
)

# ---------- 3) Optional notebook preview (rounded) ----------
preview_df = table_df.copy()
for c in ["p_out", "p_in", "analytical_Sw", "p2_5", "p97_5", "bias"]:
    preview_df[c] = preview_df[c].round(3)
display(preview_df)

# ---------- 4) Prepare LaTeX-friendly column names ----------
latex_df = table_df.rename(columns={
    "p_out": r"$p_{\mathrm{out}}$",
    "p_in": r"$p_{\mathrm{in}}$",
    "analytical_Sw": r"$S_w^{\mathrm{true}}$",
    "p2_5": r"$\hat S_w^{(2.5\%)}$",
    "p97_5": r"$\hat S_w^{(97.5\%)}$",
    "bias": r"$\overline{\hat S_w}-S_w^{\mathrm{true}}$",
})

# ---------- 5) Column formatters (3 decimals) ----------
formatters = {
    r"$p_{\mathrm{out}}$": lambda x: f"{x:.3f}",
    r"$p_{\mathrm{in}}$": lambda x: f"{x:.3f}",
    r"$S_w^{\mathrm{true}}$": lambda x: f"{x:.3f}",
    r"$\hat S_w^{(2.5\%)}$": lambda x: f"{x:.3f}",
    r"$\hat S_w^{(97.5\%)}$": lambda x: f"{x:.3f}",
    r"$\overline{\hat S_w}-S_w^{\mathrm{true}}$": lambda x: f"{x:.3f}",
}

# ---------- 6) Generate LaTeX ----------
base_latex = latex_df.to_latex(
    index=False,
    escape=False,                 # keep math headers
    formatters=formatters,
    column_format="cccccc",       # 6 centered columns
    caption=("Per-configuration summary: analytical $S_w$, empirical 2.5th and 97.5th percentiles, "
             "and bias."),
    label="tab:sim_percentile_bias",
)

# Optional compact styling for two-column PDFs
latex_code = base_latex.replace(
    r"\begin{table}",
    r"\begin{table}" + "\n" + r"\setlength{\tabcolsep}{3pt}" + "\n" + r"\scriptsize"
)

print(latex_code)

# ---------- 7) Optional: save to .tex ----------
# with open("sim_percentile_bias_table.tex", "w") as f:
#     f.write(latex_code)
# print("Saved: sim_percentile_bias_table.tex")

In [ ]:
# ============================================================
# Plotting block: three independent charts (one per p_out)
# Assumes results_df exists with columns:
# p_out, p_in, true_Sw, mean_hat_Sw, emp_ci_low, emp_ci_high
# ============================================================

for p_out in p_out_values:
    sub = (
        results_df[results_df["p_out"] == p_out]
        .sort_values("p_in")
        .copy()
    )

    plt.figure(figsize=(7, 5))

    # True analytic line
    plt.plot(
        sub["p_in"],
        sub["true_Sw"],
        label="True $S_w$",
        linewidth=2.2
    )

    # Empirical mean estimate with visible triangle markers
    plt.plot(
        sub["p_in"],
        sub["mean_hat_Sw"],
        label=r"Empirical mean $\hat S_w$",
        linestyle="--",
        linewidth=1.8,
        marker="^",
        markersize=8,
        markeredgewidth=0.8
    )

    # Empirical 95% interval band (2.5th to 97.5th percentile)
    plt.fill_between(
        sub["p_in"],
        sub["emp_ci_low"],
        sub["emp_ci_high"],
        alpha=0.20,
        label="Empirical 95% interval"
    )

    plt.title(f"Value-Weighted Segmentedness vs Within-Block Connectivity\n(p_out = {p_out})")
    plt.xlabel("Within-block connectivity (p_in)")
    plt.ylabel("Value-weighted segmentedness")
    plt.grid(alpha=0.30)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.ylim(0, 1)
    plt.show()